In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# hybrid delegator structure
#TODO train MNB, 
#TODO implement llm router (few-shot), using same example 
#pick enough samples (>35 to 100) from the propulation  
class HybridDelegator:
    def __init__(self, threshold=0.2):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.clf = MultinomialNB() # 
        self.threshold = threshold # Min difference between top 2 classes

    def route_query(self, user_query):
        # 1. Transform and Predict Probabilities
        vectorized_input = self.vectorizer.transform([user_query])
        probs = self.clf.predict_proba(vectorized_input)[0]
        
        # 2. Rank Intent Indices
        top_indices = np.argsort(probs)[::-1]
        top_idx, second_idx = top_indices[0], top_indices[1]
        
        # 3. Check Confidence Margin
        margin = probs[top_idx] - probs[second_idx]
        
        if margin >= self.threshold:
            return self.clf.classes_[top_idx] # Confident MNB route
        else:
            return self.invoke_llm_router(user_query) # Fallback to LLM